In [0]:
from faker import Faker
import random
from datetime import datetime, timedelta
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DateType
 
# Initialize Faker
fake = Faker()
 
# Define the schema for the Students table
schema = StructType([
    StructField("student_id", StringType(), False),
    StructField("application_id", StringType(), False),
    StructField("first_name", StringType(), True),
    StructField("last_name", StringType(), True),
    StructField("date_of_birth", DateType(), True),
    StructField("gender", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("admission_date", DateType(), True),
    StructField("program_id", StringType(), True),
    StructField("status", StringType(), True)
])
 
# Generate 3500 random student records
students_data = []
status_options = ['Active', 'Graduated', 'Dropped']
gender_options = ['Male', 'Female', 'Other']
 
for i in range(1, 3501):
    student_id = f"STU{1000 + i}"
    application_id = f"APP{str(i).zfill(4)}"
    first_name = fake.first_name()
    last_name = fake.last_name()
    date_of_birth = fake.date_of_birth(minimum_age=18, maximum_age=35)
    gender = random.choice(gender_options)
    email = fake.email()
    phone = fake.phone_number()
    admission_date = fake.date_between(start_date='-5y', end_date='today')
    program_id = f"PGM{str(random.randint(1, 50)).zfill(3)}"
    status = random.choice(status_options)
   
    students_data.append((
        student_id,
        application_id,
        first_name,
        last_name,
        date_of_birth,
        gender,
        email,
        phone,
        admission_date,
        program_id,
        status
    ))
 
# Create DataFrame
df = spark.createDataFrame(students_data, schema=schema)
 
# Display sample records
print(f"Total records generated: {df.count()}")
df.show(10)
 
# Create the Students table in the specific catalog and schema location
df.write.format("delta").mode("overwrite").saveAsTable("university_lakehouse.bronze.bronze_student")
 
print("bronze_student table created successfully in university_lakehouse.bronze with 3500 records!")

In [0]:
# Write the students DataFrame to Delta table
df.write.format("delta").mode("overwrite").saveAsTable("university_lakehouse.bronze.bronze_student")

print("Delta table created successfully at university_lakehouse.bronze.bronze_student")

In [0]:
%sql
select * from university_lakehouse.bronze.bronze_student

In [0]:
# Cell 1: Install Faker
%pip install Faker


In [0]:
# Cell 2: Imports and generate data
from faker import Faker
import random
from datetime import date
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, StringType, DateType, IntegerType

fake = Faker("en_IN")

# 20 program IDs matching your PROGRAMS table
program_ids = [f"P{i:03d}" for i in range(1, 21)]

statuses = ["Submitted", "Accepted", "Rejected", "Waitlisted"]
genders = ["Male", "Female"]

# Define schema
schema = StructType([
    StructField("application_id", StringType(), False),
    StructField("applicant_name", StringType(), True),
    StructField("email", StringType(), True),
    StructField("phone", StringType(), True),
    StructField("gender", StringType(), True),
    StructField("city", StringType(), True),
    StructField("state", StringType(), True),
    StructField("application_date", DateType(), True),
    StructField("program_id", StringType(), True),
    StructField("entrance_score", IntegerType(), True),
    StructField("application_status", StringType(), True)
])

start_date = date(2026, 1, 1)
end_date = date(2026, 8, 31)

records = []

for i in range(1, 5001):
    application_id = f"APP{i:04d}"
    applicant_name = fake.name()
    email = fake.email()
    phone = fake.msisdn()[:10]
    gender = random.choice(genders)
    city = fake.city()
    state = fake.state()

    application_date = fake.date_between(
        start_date=start_date,
        end_date=end_date
    )

    program_id = random.choice(program_ids)
    entrance_score = random.randint(50, 100)

    application_status = random.choices(
        statuses,
        weights=[10, 40, 25, 25],
        k=1
    )[0]

    records.append((
        application_id,
        applicant_name,
        email,
        phone,
        gender,
        city,
        state,
        application_date,
        program_id,
        entrance_score,
        application_status
    ))

print(f"Generated {len(records)} records")


In [0]:
# Cell 3: Create DataFrame
df = spark.createDataFrame(records, schema=schema)

print(f"Total records generated: {df.count()}")
df.show(10)


In [0]:
# Cell 4: Write to Delta table at your catalog/schema location
df.write.format("delta").mode("overwrite").saveAsTable("university_lakehouse.bronze.bronze_applications")

print("Delta table created successfully at university_lakehouse.bronze.bronze_applications")


In [0]:
%sql
select * from university_lakehouse.bronze.bronze_applications